# Lab 3 — ColBERT: Retrieval Granular com RAGatouille
## Comparativo nDCG@10: ColBERT vs BGE-M3 em Corpus Jurídico

**Aula 11 · MBA RAG & CAG Aplicados a Direito e Segurança Pública**

**Objetivo:** Indexar um corpus jurídico com ColBERT via RAGatouille, executar 20 queries técnicas e comparar a métrica nDCG@10 com o bi-encoder BGE-M3.

**Duração estimada:** 60 minutos

---

### O que é nDCG@10?

**nDCG** (Normalized Discounted Cumulative Gain) mede a qualidade do ranking:
- Documentos relevantes no topo = score alto
- Documentos relevantes no fim = score baixo (desconto logarítmico)
- Normalizado: varia de 0.0 a 1.0

```
nDCG@10 = DCG@10 / IDCG@10

DCG@10 = Σ (rel_i / log2(i+1))  para i = 1..10
          posição 1: rel / log2(2) = rel / 1.0
          posição 2: rel / log2(3) = rel / 1.58
          posição 10: rel / log2(11) = rel / 3.46
```

## Etapa 0 — Instalação

In [ ]:
!pip install -q ragatouille==0.0.8
!pip install -q sentence-transformers==3.0.0
!pip install -q numpy==1.26.4

print("✅ Dependências instaladas!")

## Etapa 1 — Corpus Jurídico para Benchmark

In [ ]:
import numpy as np
from pathlib import Path

# Corpus de 30 documentos jurídicos para benchmark
# Abrange temas de Direito Penal e Segurança Pública

corpus = [
    # Documentos sobre ROUBO e afins
    {"id": "D01", "texto": "Roubo (Art. 157 CP): subtração de coisa móvel mediante grave ameaça ou violência. Pena: 4 a 10 anos. Majorado com arma de fogo: 2/3 da pena. Roubo qualificado com morte: latrocínio (Art. 157, §3º, II)."},
    {"id": "D02", "texto": "Furto qualificado (Art. 155, §4º CP): com destruição de obstáculo, abuso de confiança, mediante fraude, escalada ou destreza. Pena: 2 a 8 anos. Furto simples: 1 a 4 anos. Difere do roubo pela ausência de violência."},
    {"id": "D03", "texto": "Latrocínio: roubo com resultado morte. Classificado pelo STF como crime hediondo. Pena: 20 a 30 anos de reclusão. Consuma-se com a morte, independentemente da subtração do bem. Súmula 610 STF: há latrocínio quando o homicídio se consuma."},
    {"id": "D04", "texto": "Extorsão (Art. 158 CP): constranger mediante violência ou ameaça a fazer, tolerar ou omitir algo, com vantagem econômica indevida. Pena: 4 a 10 anos. Distingue-se do roubo: na extorsão a vantagem depende do comportamento da vítima."},
    # Documentos sobre TRÁFICO
    {"id": "D05", "texto": "Tráfico de drogas (Art. 33 Lei 11.343/06): importar, exportar, vender, guardar, transportar drogas sem autorização. Pena: 5 a 15 anos. Crime inafiançável e insuscetível de sursis, graça ou anistia."},
    {"id": "D06", "texto": "Tráfico privilegiado (Art. 33, §4º Lei 11.343/06): agente primário, bons antecedentes, não integrante de organização criminosa, não dedicado a atividades criminosas. Redução de pena: 1/6 a 2/3."},
    {"id": "D07", "texto": "Associação ao tráfico (Art. 35 Lei 11.343/06): associação de 2 ou mais pessoas para praticar reiteradamente o tráfico. Pena: 3 a 10 anos. Distingue-se do concurso de agentes: exige estabilidade e permanência."},
    {"id": "D08", "texto": "Uso de drogas (Art. 28 Lei 11.343/06): adquirir para consumo pessoal. Não é crime, é infração administrativa. Penas: advertência, prestação de serviços à comunidade, medida educativa. Porte para uso versus tráfico: análise do conjunto probatório."},
    # Documentos sobre ORGANIZAÇÕES CRIMINOSAS
    {"id": "D09", "texto": "Organização criminosa (Lei 12.850/13): associação de 4 ou mais pessoas estruturalmente ordenada para obter vantagem de qualquer natureza mediante prática de infrações penais de pena máxima superior a 4 anos. Pena: 3 a 8 anos."},
    {"id": "D10", "texto": "Colaboração premiada (Art. 4º Lei 12.850/13): acordo entre Ministério Público e colaborador. Benefícios: redução de 1/3 a 2/3 da pena, substituição, perdão judicial. Eficácia probatória: corroboração com outros elementos."},
    {"id": "D11", "texto": "Infiltração policial (Art. 10 Lei 12.850/13): infiltração de agentes de polícia em tarefas de investigação de organizações criminosas. Autorização judicial obrigatória. Prazo: até 6 meses, renovável por igual período."},
    {"id": "D12", "texto": "Interceptação telemática (Lei 9.296/96): interceptação de comunicações em sistemas de informática e telemática. Requisitos: indícios razoáveis, impossibilidade de outros meios, crime punido com reclusão. Prazo: 15 dias, renovável."},
    # Documentos sobre CRIMES HEDIONDOS
    {"id": "D13", "texto": "Crimes hediondos (Lei 8.072/90): rol taxativo incluindo latrocínio, extorsão com morte, estupro, tráfico, genocídio, tortura, terrorismo. Vedados: fiança, graça, anistia. Progressão de regime: após cumprimento de 40% (réu primário) ou 60% (reincidente)."},
    {"id": "D14", "texto": "Tortura (Lei 9.455/97): constranger alguém com sofrimento físico ou mental para obter informação, confissão, ou para aplicar castigo. Pena: 2 a 8 anos. Crime inafiançável, insuscetível de graça. Agente público: pena aumentada."},
    # Documentos sobre PRISÕES E MEDIDAS CAUTELARES
    {"id": "D15", "texto": "Prisão preventiva (Art. 312 CPP): garantia da ordem pública, ordem econômica, conveniência da instrução, assegurar aplicação da lei penal. Requisito: prova da existência do crime e indício de autoria. Vedada para crimes com pena máxima ≤ 4 anos."},
    {"id": "D16", "texto": "Habeas corpus (Art. 5º, LXVIII CF): instrumento constitucional para proteger a liberdade de locomoção quando ameaçada por ilegalidade ou abuso de poder. HC preventivo: salvo-conduto. HC liberatório: relaxamento da prisão."},
    {"id": "D17", "texto": "Prisão em flagrante (Art. 301 CPP): qualquer cidadão pode prender quem esteja em flagrante delito. Espécies: próprio, impróprio, presumido, esperado. Lavratura do APF: 24 horas. Conversão em preventiva: audiência de custódia."},
    {"id": "D18", "texto": "Audiência de custódia (Resolução CNJ 213/2015): apresentação do preso ao juiz em até 24 horas após a prisão. Finalidade: verificar legalidade da prisão, identificar violência ou tortura, decidir sobre a manutenção ou relaxamento da prisão."},
    # Documentos sobre ARMAS
    {"id": "D19", "texto": "Estatuto do Desarmamento (Lei 10.826/03): porte ilegal de arma de fogo (Art. 14): pena 2 a 4 anos. Posse irregular (Art. 12): 1 a 3 anos. Comércio ilegal (Art. 17): 6 a 12 anos. Arma de uso restrito: penas dobradas."},
    {"id": "D20", "texto": "Tráfico de armas (Art. 18 Lei 10.826/03): importar, exportar, comerciar armas sem autorização. Pena: 8 a 16 anos. Crime formal. A internacionalidade não é elemento, mas majora a pena. Investigação frequentemente vinculada a organizações criminosas."},
    # Documentos sobre PROCEDIMENTO POLICIAL
    {"id": "D21", "texto": "Inquérito policial (Art. 4º CPP): procedimento administrativo inquisitório presidido pela autoridade policial. Características: inquisitório, sigiloso, escrito, indisponível. Prazo: 10 dias (preso), 30 dias (solto), prorrogável."},
    {"id": "D22", "texto": "Busca e apreensão (Art. 240 CPP): domiciliar (mandado judicial) e pessoal (prescinde de mandado em flagrante ou fundada suspeita). Inviolabilidade de domicílio (Art. 5º, XI CF): exceções: flagrante, desastre, socorro, determinação judicial diurna."},
    {"id": "D23", "texto": "Cadeia de custódia (Art. 158-A CPP - Pacote Anticrime): conjunto de procedimentos para preservação da integridade dos vestígios. Elementos: reconhecimento, isolamento, fixação, coleta, acondicionamento, transporte, recebimento, processamento, armazenamento, descarte."},
    # Documentos sobre INTELIGÊNCIA
    {"id": "D24", "texto": "Atividade de inteligência policial: produção de conhecimento para subsidiar decisões de segurança pública. Ciclo: coleta, processamento, análise, disseminação. Diferencia-se da investigação criminal: busca antecipar ameaças, não apurar fatos passados."},
    {"id": "D25", "texto": "SISBIN (Sistema Brasileiro de Inteligência Lei 9.883/99): coordenado pela ABIN. Integra atividades de inteligência dos órgãos públicos. Princípios: objetividade, controle, imparcialidade, sigilo. Acesso restrito a agentes credenciados."},
    # Documentos adicionais
    {"id": "D26", "texto": "Interrogatório do acusado (Art. 185 CPP): direito ao silêncio (Art. 5º, LXIII CF). Presença de advogado obrigatória. Vedada a condução coercitiva para interrogatório (ADPF 395 e 444 STF). Confissão: valor probatório relativo."},
    {"id": "D27", "texto": "Lei Maria da Penha (Lei 11.340/06): violência doméstica e familiar contra a mulher. Medidas protetivas (Art. 22): afastamento, proibição de aproximação, suspensão de armas. Delegacias especializadas (DEAM). Crime de feminicídio: Art. 121, §2º, VI CP."},
    {"id": "D28", "texto": "Estatuto da Criança e do Adolescente (Lei 8.069/90): ato infracional análogo ao crime: medidas socioeducativas. Internação: máximo 3 anos, liberação compulsória aos 21 anos. Semiliberdade, liberdade assistida, prestação de serviços à comunidade."},
    {"id": "D29", "texto": "Lavagem de capitais (Lei 9.613/98): ocultar ou dissimular bens provenientes de crimes. Fases: colocação, ocultação, integração. Pena: 3 a 10 anos. Crimes antecedentes: qualquer infração penal. Delação premiada aplicável."},
    {"id": "D30", "texto": "Abuso de autoridade (Lei 13.869/19): conduta do agente público que detenha ou prive a liberdade de alguém ilegalmente, submeta a situação de humilhação ou constrangimento, ou retarde injustificadamente operação de legítima defesa. Pena: detenção e multa."},
]

print(f"📚 Corpus carregado: {len(corpus)} documentos jurídicos")
print(f"Temas: Crimes contra patrimônio, Tráfico, ORCRIM, Crimes hediondos, Prisões, Armas, Procedimento")

## Etapa 2 — Gabarito de Relevância (Ground Truth)

In [ ]:
# 20 queries técnicas com gabarito de relevância
# relevancia: {doc_id: score} onde 2=muito relevante, 1=relevante, 0=irrelevante

queries_benchmark = [
    {
        "id": "Q01",
        "query": "requisitos tráfico privilegiado pena redução",
        "relevancia": {"D06": 2, "D05": 1, "D07": 1, "D08": 1}
    },
    {
        "id": "Q02",
        "query": "roubo arma de fogo pena majorada",
        "relevancia": {"D01": 2, "D19": 1, "D20": 1}
    },
    {
        "id": "Q03",
        "query": "prisão preventiva requisitos fundamento",
        "relevancia": {"D15": 2, "D16": 1, "D17": 1, "D18": 1}
    },
    {
        "id": "Q04",
        "query": "organização criminosa quatro pessoas estruturada",
        "relevancia": {"D09": 2, "D10": 1, "D11": 1, "D07": 1}
    },
    {
        "id": "Q05",
        "query": "latrocínio roubo resultado morte hediondo",
        "relevancia": {"D03": 2, "D13": 1, "D01": 1}
    },
    {
        "id": "Q06",
        "query": "colaboração premiada benefícios redução pena",
        "relevancia": {"D10": 2, "D09": 1, "D29": 1}
    },
    {
        "id": "Q07",
        "query": "habeas corpus liberdade locomoção ilegalidade",
        "relevancia": {"D16": 2, "D15": 1, "D17": 1}
    },
    {
        "id": "Q08",
        "query": "interceptação comunicação telemática prazo judicial",
        "relevancia": {"D12": 2, "D11": 1, "D25": 1}
    },
    {
        "id": "Q09",
        "query": "busca apreensão domiciliar mandado judicial inviolabilidade",
        "relevancia": {"D22": 2, "D21": 1, "D23": 1}
    },
    {
        "id": "Q10",
        "query": "cadeia de custódia vestígios preservação",
        "relevancia": {"D23": 2, "D21": 1, "D22": 1}
    },
    {
        "id": "Q11",
        "query": "crimes hediondos rol progressão regime",
        "relevancia": {"D13": 2, "D03": 1, "D14": 1, "D05": 1}
    },
    {
        "id": "Q12",
        "query": "porte ilegal arma fogo pena estatuto desarmamento",
        "relevancia": {"D19": 2, "D20": 1, "D01": 1}
    },
    {
        "id": "Q13",
        "query": "infiltração policial organização criminosa autorização judicial",
        "relevancia": {"D11": 2, "D09": 1, "D12": 1}
    },
    {
        "id": "Q14",
        "query": "audiência custódia prazo apresentação preso juiz",
        "relevancia": {"D18": 2, "D17": 1, "D15": 1}
    },
    {
        "id": "Q15",
        "query": "lavagem capitais fases ocultar dissimular",
        "relevancia": {"D29": 2, "D09": 1, "D10": 1}
    },
    {
        "id": "Q16",
        "query": "violência doméstica medidas protetivas afastamento",
        "relevancia": {"D27": 2, "D14": 1}
    },
    {
        "id": "Q17",
        "query": "ato infracional adolescente medidas socioeducativas internação",
        "relevancia": {"D28": 2}
    },
    {
        "id": "Q18",
        "query": "tortura agente público sofrimento físico mental",
        "relevancia": {"D14": 2, "D13": 1, "D30": 1}
    },
    {
        "id": "Q19",
        "query": "inquérito policial prazo sigiloso inquisitório",
        "relevancia": {"D21": 2, "D22": 1}
    },
    {
        "id": "Q20",
        "query": "abuso autoridade policial privação liberdade humilhação",
        "relevancia": {"D30": 2, "D14": 1, "D16": 1}
    },
]

print(f"✅ Benchmark configurado: {len(queries_benchmark)} queries com gabarito de relevância")

## Etapa 3 — Função nDCG@k

In [ ]:
import math

def calcular_ndcg(ids_recuperados: list[str], relevancia_gabarito: dict, k: int = 10) -> float:
    """
    Calcula nDCG@k dado os IDs dos documentos recuperados e o gabarito.
    
    Args:
        ids_recuperados: Lista de IDs na ordem retornada pelo sistema
        relevancia_gabarito: Dict {doc_id: score_relevancia}
        k: Posição de corte
    
    Returns:
        nDCG@k entre 0.0 e 1.0
    """
    ids_top_k = ids_recuperados[:k]
    
    # DCG: gain descontado
    dcg = 0.0
    for posicao, doc_id in enumerate(ids_top_k, start=1):
        rel = relevancia_gabarito.get(doc_id, 0)
        dcg += rel / math.log2(posicao + 1)
    
    # IDCG: ideal (documentos relevantes no topo)
    scores_ideais = sorted(relevancia_gabarito.values(), reverse=True)[:k]
    idcg = sum(
        rel / math.log2(pos + 2)
        for pos, rel in enumerate(scores_ideais)
    )
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg


# Teste da função
ids_perfeito = ["D06", "D05", "D07", "D08", "D01", "D09", "D10", "D11", "D12", "D13"]
relevancia_q01 = {"D06": 2, "D05": 1, "D07": 1, "D08": 1}

ndcg_perfeito = calcular_ndcg(ids_perfeito, relevancia_q01, k=10)
print(f"Teste nDCG@10 (ranking perfeito): {ndcg_perfeito:.4f}")

ids_invertido = ["D13", "D12", "D11", "D10", "D09", "D01", "D08", "D07", "D05", "D06"]
ndcg_invertido = calcular_ndcg(ids_invertido, relevancia_q01, k=10)
print(f"Teste nDCG@10 (ranking invertido): {ndcg_invertido:.4f}")
print("\n✅ Função nDCG implementada corretamente")

## Etapa 4 — Benchmark BGE-M3 (Bi-Encoder)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("📥 Carregando BGE-M3...")
modelo_bge = SentenceTransformer("BAAI/bge-m3")
print("✅ BGE-M3 carregado")

# Indexar corpus com BGE-M3
textos_corpus = [d["texto"] for d in corpus]
ids_corpus = [d["id"] for d in corpus]

print("\n🔄 Gerando embeddings do corpus com BGE-M3...")
embeddings_corpus_bge = modelo_bge.encode(
    textos_corpus,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=8
)

print(f"✅ Corpus indexado: {embeddings_corpus_bge.shape}")


def busca_bge(query: str, k: int = 10) -> list[str]:
    """Busca por similaridade cosseno com BGE-M3."""
    q_emb = modelo_bge.encode(query, normalize_embeddings=True)
    scores = embeddings_corpus_bge @ q_emb
    indices_ordenados = np.argsort(scores)[::-1][:k]
    return [ids_corpus[i] for i in indices_ordenados]


# Executar benchmark BGE-M3
print("\n🔄 Executando benchmark BGE-M3 (20 queries)...")
resultados_bge = []

for q in queries_benchmark:
    ids_recuperados = busca_bge(q["query"], k=10)
    ndcg = calcular_ndcg(ids_recuperados, q["relevancia"], k=10)
    resultados_bge.append({"id": q["id"], "ndcg": ndcg})

ndcg_medio_bge = np.mean([r["ndcg"] for r in resultados_bge])
print(f"\n✅ BGE-M3 - nDCG@10 médio: {ndcg_medio_bge:.4f}")

## Etapa 5 — Benchmark ColBERT com RAGatouille

In [ ]:
from ragatouille import RAGPretrainedModel
from pathlib import Path

print("📥 Carregando ColBERTv2 via RAGatouille...")
print("   (Download do modelo pode demorar 2-3 min)")

# Inicializar ColBERT
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
print("✅ ColBERTv2 carregado!")

# Indexar corpus
INDEX_PATH = "/content/colbert_index_juridico"
print("\n🔄 Indexando corpus com ColBERT...")
print("   (Processo mais lento que bi-encoder — armazena token-level embeddings)")

RAG.index(
    collection=textos_corpus,
    document_ids=ids_corpus,
    index_name="juridico_colbert",
    max_document_length=512,
    split_documents=True,
)

print("✅ Índice ColBERT criado!")


def busca_colbert(query: str, k: int = 10) -> list[str]:
    """Busca ColBERT com MaxSim late interaction."""
    resultados = RAG.search(query=query, k=k)
    # RAGatouille retorna lista de dicts com 'document_id'
    return [r.get("document_id", r.get("content", "")[:4]) for r in resultados]


# Executar benchmark ColBERT
print("\n🔄 Executando benchmark ColBERT (20 queries)...")
resultados_colbert = []

for q in queries_benchmark:
    try:
        ids_recuperados = busca_colbert(q["query"], k=10)
        ndcg = calcular_ndcg(ids_recuperados, q["relevancia"], k=10)
    except Exception as e:
        print(f"  ⚠️ Erro na query {q['id']}: {e}")
        ndcg = 0.0
    resultados_colbert.append({"id": q["id"], "ndcg": ndcg})

ndcg_medio_colbert = np.mean([r["ndcg"] for r in resultados_colbert])
print(f"\n✅ ColBERT - nDCG@10 médio: {ndcg_medio_colbert:.4f}")

## Etapa 6 — Comparativo Final

In [ ]:
# Comparativo detalhado query a query

print("=" * 75)
print("COMPARATIVO nDCG@10: BGE-M3 vs ColBERT")
print("=" * 75)
print(f"{'Query ID':<10} {'Query (resumo)':<35} {'BGE-M3':>8} {'ColBERT':>8} {'Ganho':>8}")
print("-" * 75)

ganhos = []
for bge, colbert, q in zip(resultados_bge, resultados_colbert, queries_benchmark):
    ganho = colbert["ndcg"] - bge["ndcg"]
    ganhos.append(ganho)
    query_resumo = q["query"][:34]
    indicador = "↑" if ganho > 0.01 else ("↓" if ganho < -0.01 else "=")
    print(f"{q['id']:<10} {query_resumo:<35} {bge['ndcg']:>8.4f} {colbert['ndcg']:>8.4f} {indicador} {ganho:>+6.4f}")

print("-" * 75)
print(f"{'MÉDIA':<10} {'':35} {ndcg_medio_bge:>8.4f} {ndcg_medio_colbert:>8.4f} {np.mean(ganhos):>+8.4f}")

print("\n=" * 75)
print("RESUMO EXECUTIVO")
print("=" * 75)

melhora_pct = (ndcg_medio_colbert / ndcg_medio_bge - 1) * 100 if ndcg_medio_bge > 0 else 0
queries_colbert_ganhou = sum(1 for g in ganhos if g > 0.01)
queries_bge_ganhou = sum(1 for g in ganhos if g < -0.01)
queries_empate = len(ganhos) - queries_colbert_ganhou - queries_bge_ganhou

print(f"BGE-M3  nDCG@10 médio: {ndcg_medio_bge:.4f}")
print(f"ColBERT nDCG@10 médio: {ndcg_medio_colbert:.4f}")
print(f"Melhora ColBERT: {melhora_pct:+.1f}%")
print(f"\nColBERT venceu em: {queries_colbert_ganhou}/20 queries")
print(f"BGE-M3  venceu em: {queries_bge_ganhou}/20 queries")
print(f"Empate:            {queries_empate}/20 queries")

print("""
💡 Interpretação:
   ColBERT tende a superar BGE-M3 em queries com:
   • Termos técnicos específicos (art. 33, §4º)
   • Múltiplos conceitos combinados (tráfico + privilegiado + pena)
   • Distinções sutis entre conceitos similares (roubo vs extorsão)
   
   BGE-M3 pode ser suficiente para:
   • Buscas de alto nível temático ("tudo sobre tráfico")
   • Consultas simples de um único conceito
   • Ambientes com restrição de recursos (índice menor)
""")

## Exercício Prático

> **Exercício:** Adicione 5 novos documentos ao corpus (sobre temas de sua escolha) e 3 novas queries ao benchmark. Execute o comparativo e analise se o padrão de resultados (ColBERT × BGE-M3) se mantém ou muda. Documente sua hipótese sobre o porquê.

## Pergunta de Reflexão

> **Reflexão:** ColBERT requer um índice 3–5× maior que bi-encoders. Em um sistema de uma delegacia com limitação de hardware, como você decidiria entre usar ColBERT (maior precisão, mais memória) ou BGE-M3 (menor precisão, menor custo)? Que critérios objetivos você usaria?

---

**Próximo:** Lab 4 — Time-Aware RAG com Decay Temporal no OpenSearch